### Harle et al Screen

This notebook preprocesses CRISPR screen data from *Harle et al., 2025*  
It loads log-fold change (LFC) and false discovery rate (FDR) scores from the paper’s supplementary tables, cleans and formats them, and prepares the dataset for downstream feature annotation and training the Random Forest classifier.

**Inputs:**  
- CSV files containing LFC and FDR scores from the *Harle et al., 2025* supplementary materials.

**Outputs:**  
- A cleaned and merged dataset saved as a CSV file, ready for feature annotation and model training.


In [1]:
# import modules
import os
import pandas as pd
import numpy as np
from natsort import natsorted

In [2]:
# set the base directory for the project
cwd = os.getcwd()
BASE_DIR = os.path.abspath(os.path.join(cwd, "..", ".."))

# build paths inside the repo
get_data_path = lambda folders, fname: os.path.normpath(
    os.path.join(BASE_DIR, *folders, fname)
)

file_path_genenames = get_data_path(['data', 'input', 'other'], 'approved_and_previous_symbols.csv')
file_path_harle_screen_SL = get_data_path(['data', 'input', 'CRISPR_screens'], 'Harle_TableS4.xlsx')
file_path_harle_screen_binary = get_data_path(['data', 'input', 'CRISPR_screens'], 'Harle_TableS5.xlsx')

file_path_processed_harle_df = get_data_path(['data', 'output', 'processed_CRISPR_screens'], 'processed_harle_df.csv')

In [3]:
def label_sl_per_cell_line(
    df: pd.DataFrame,
    gi_threshold: float = -0.5,
    fdr_threshold: float = 0.01,
) -> pd.DataFrame:

    required = {
        "mean_norm_gi", "fdr", "is_bassik_hit",
        "targetA__is_single_depleted", "targetB__is_single_depleted",
    }
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"TableS4 is missing expected columns: {missing}")
 
    df = df.copy()
    df["is_sl_hit"] = (
        (df["mean_norm_gi"] < gi_threshold)
        & (df["fdr"] < fdr_threshold)
        & (df["is_bassik_hit"] == 1)
        & (df["targetA__is_single_depleted"] == 0)
        & (df["targetB__is_single_depleted"] == 0)
    )
    return df

In [4]:
s4 = pd.read_excel(file_path_harle_screen_SL)
labelled_harle_screen = label_sl_per_cell_line(s4)

In [5]:
labelled_harle_screen = labelled_harle_screen[[
                      'sorted_gene_pair',
                      'targetA', 'targetB',
                      'cell_line_label',
                      'depMapID',
                      'mean_norm_gi',
                      'fdr',
                      'is_bassik_hit',
                      'targetA__is_single_depleted',
                      'targetB__is_single_depleted',
                      'is_sl_hit']].copy()

labelled_harle_screen = labelled_harle_screen.rename(columns={
    'cell_line_label': 'cell_line',
    'depMapID': 'DepMap_ID'})

In [6]:
print(f'number of unique gene pairs: {labelled_harle_screen.sorted_gene_pair.nunique()}')
print(f'number of unique cell lines: {labelled_harle_screen.cell_line.nunique()}')
print(f'number of SL: {labelled_harle_screen.is_sl_hit.value_counts().get(True, 0)}')
print(f'number of non-SL: {labelled_harle_screen.is_sl_hit.value_counts().get(False, 0)}')
labelled_harle_screen[:3]

number of unique gene pairs: 472
number of unique cell lines: 27
number of SL: 882
number of non-SL: 11862


,sorted_gene_pair,targetA,targetB,cell_line,DepMap_ID,mean_norm_gi,fdr,is_bassik_hit,targetA__is_single_depleted,targetB__is_single_depleted,is_sl_hit
0,SEC23A|SEC23B,SEC23A,SEC23B,A-375,ACH-000219,-1.375503,7.126670e-18,1,0,0,True
1,SLC25A28|SLC25A37,SLC25A37,SLC25A28,A-375,ACH-000219,-1.251507,1.417212e-15,1,0,0,True
2,ASF1A|ASF1B,ASF1B,ASF1A,A-375,ACH-000219,-1.520299,1.599033e-15,1,0,0,True


In [7]:
harle_screen_binary = pd.read_excel(file_path_harle_screen_binary)
harle_screen_binary['sorted_gene_pair'] = harle_screen_binary['sorted_gene_pair'].astype(str).str.strip()
harle_screen_binary_df = harle_screen_binary[harle_screen_binary.columns[:28]].copy()
#harle_screen_binary_df = harle_screen_binary_df.drop(columns=['C092'])

In [8]:
def label_sl_summary(
    df: pd.DataFrame,
    min_cell_lines: int = 1,
    cell_line_cols: list = None,
) -> pd.DataFrame:

    if cell_line_cols is None:
        cell_line_cols = CELL_LINE_COLUMNS
 
    available = [c for c in cell_line_cols if c in df.columns]
    missing = set(cell_line_cols) - set(available)
    if missing:
        print(f"Warning: {len(missing)} cell line columns not found and skipped: {missing}")
 
    df = df.copy()
    df["n_cell_lines_hit"] = df[available].sum(axis=1)
    df["is_sl_hit"] = df["n_cell_lines_hit"] >= min_cell_lines
    return df

In [9]:
harle_screen_binary_summary = label_sl_summary(harle_screen_binary_df, 
                                               min_cell_lines=1, 
                                               cell_line_cols=harle_screen_binary_df.columns[1:28])

In [10]:
print(f'number of unique gene pairs: {harle_screen_binary_summary.sorted_gene_pair.nunique()}')
print(f'number of SL: {harle_screen_binary_summary.is_sl_hit.value_counts().get(True, 0)}')
print(f'number of non-SL: {harle_screen_binary_summary.is_sl_hit.value_counts().get(False, 0)}')
harle_screen_binary_summary[:3]

number of unique gene pairs: 472
number of SL: 117
number of non-SL: 355


,sorted_gene_pair,A-375,A2058,A549,AsPC-1,BxPC-3,C092,Capan-1,CFPAC-1,CHL-1,...,NCI-H1568,NCI-H1975,NCI-H23,SK-MEL-2,SK-MEL-28,SK-MEL-5,SK-MES-1,SU.86.86,n_cell_lines_hit,is_sl_hit
0,ABCC3|ATP5PB,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,False
1,ABL1|ABL2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,False
2,ABT1|LGALS9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,False


In [11]:
cell_cols = [c for c in harle_screen_binary_summary.columns
             if c not in ('sorted_gene_pair', 'n_cell_lines_hit', 'is_sl_hit')]

robust_harle_screen = (
    harle_screen_binary_summary
    .drop(columns=['is_sl_hit'])
    .melt(
        id_vars=['sorted_gene_pair', 'n_cell_lines_hit'],
        value_vars=cell_cols,
        var_name='cell_line',
        value_name='is_sl_hit'
    )
    .rename(columns={'sorted_gene_pair': 'genepair'})
    [['genepair', 'cell_line', 'n_cell_lines_hit', 'is_sl_hit']]
    .sort_values(['genepair', 'cell_line'])
    .reset_index(drop=True)
)

robust_harle_screen.loc[robust_harle_screen['genepair'] == 'ADSS|ADSSL1']

,genepair,cell_line,n_cell_lines_hit,is_sl_hit
270,ADSS|ADSSL1,A-375,2,0
271,ADSS|ADSSL1,A2058,2,0
272,ADSS|ADSSL1,A549,2,0
273,ADSS|ADSSL1,AsPC-1,2,0
274,ADSS|ADSSL1,BxPC-3,2,0
275,ADSS|ADSSL1,C092,2,0
276,ADSS|ADSSL1,CFPAC-1,2,0
277,ADSS|ADSSL1,CHL-1,2,0
278,ADSS|ADSSL1,COR-L23,2,0
279,ADSS|ADSSL1,Capan-1,2,0


### Map Gene Symbols to Entrez IDs

- Map original gene symbols to Entrez IDs using the combined mapping dictionary
- Convert Entrez IDs back to approved gene symbols for consistency
- Remove rows where gene symbols cannot be mapped to Entrez IDs

In [12]:
labelled_harle_screen

,sorted_gene_pair,targetA,targetB,cell_line,DepMap_ID,mean_norm_gi,fdr,is_bassik_hit,targetA__is_single_depleted,targetB__is_single_depleted,is_sl_hit
0,SEC23A|SEC23B,SEC23A,SEC23B,A-375,ACH-000219,-1.375503,7.126670e-18,1,0,0,True
1,SLC25A28|SLC25A37,SLC25A37,SLC25A28,A-375,ACH-000219,-1.251507,1.417212e-15,1,0,0,True
2,ASF1A|ASF1B,ASF1B,ASF1A,A-375,ACH-000219,-1.520299,1.599033e-15,1,0,0,True
3,EAF1|EAF2,EAF1,EAF2,A-375,ACH-000219,-1.539965,5.117539e-15,1,0,0,True
4,CCND1|CCND3,CCND3,CCND1,A-375,ACH-000219,-0.765868,6.769710e-15,0,0,1,False
...,...,...,...,...,...,...,...,...,...,...,...
12739,EPHA2|PCGF1,EPHA2,PCGF1,SK-MES-1,ACH-000665,0.918143,1.000000e+00,0,0,0,False
12740,JAG1|JAG2,JAG1,JAG2,SK-MES-1,ACH-000665,0.407344,9.957606e-01,0,0,0,False
12741,GDF15|TP53,GDF15,TP53,SK-MES-1,ACH-000665,0.053760,7.964594e-01,0,0,0,False
12742,LATS1|LATS2,LATS1,LATS2,SK-MES-1,ACH-000665,1.197589,1.000000e+00,0,0,0,False


In [13]:
# read the gene names mapping file
id_map = pd.read_csv(file_path_genenames)

# create dictionaries to map gene symbols to Entrez IDs
approved_sym_to_entrez_id = dict(zip(id_map['Approved symbol'], id_map['entrez_id']))

# create dictionaries to map previous gene symbols to Entrez IDs
id_map_cleaned = id_map.dropna(axis=0, how='any', subset=['Previous symbol', 'entrez_id']).reset_index(drop=True)
prev_sym_to_entrez_id = dict(zip(id_map_cleaned['Previous symbol'], id_map_cleaned['entrez_id']))

In [14]:
labelled_harle_screen_df = labelled_harle_screen.copy()

labelled_harle_screen_df = labelled_harle_screen_df.assign(
    A1_entrez = labelled_harle_screen_df['targetA'].map(approved_sym_to_entrez_id),
    A2_entrez = labelled_harle_screen_df['targetB'].map(approved_sym_to_entrez_id)
)

labelled_harle_screen_df['A1_entrez'] = labelled_harle_screen_df['A1_entrez'].fillna(labelled_harle_screen_df['targetA'].map(prev_sym_to_entrez_id))
labelled_harle_screen_df['A2_entrez'] = labelled_harle_screen_df['A2_entrez'].fillna(labelled_harle_screen_df['targetB'].map(prev_sym_to_entrez_id))

In [15]:
entrez_id_to_approved_sym = dict(zip(id_map['entrez_id'], id_map['Approved symbol']))
harle_screen_df = labelled_harle_screen_df.copy().assign(
    A1 = labelled_harle_screen_df['A1_entrez'].map(entrez_id_to_approved_sym),
    A2 = labelled_harle_screen_df['A2_entrez'].map(entrez_id_to_approved_sym)
)

In [16]:
print('# check the NA values in A1_entrez & A2_entrez')
display(harle_screen_df.loc[harle_screen_df['A1'].isna(), ])
display(harle_screen_df.loc[harle_screen_df['A2'].isna(), ])

# check the NA values in A1_entrez & A2_entrez


,sorted_gene_pair,targetA,targetB,cell_line,DepMap_ID,mean_norm_gi,fdr,is_bassik_hit,targetA__is_single_depleted,targetB__is_single_depleted,is_sl_hit,A1_entrez,A2_entrez,A1,A2


,sorted_gene_pair,targetA,targetB,cell_line,DepMap_ID,mean_norm_gi,fdr,is_bassik_hit,targetA__is_single_depleted,targetB__is_single_depleted,is_sl_hit,A1_entrez,A2_entrez,A1,A2


In [17]:
# natural sort genepairs

list_c = [[x, y] for x, y in zip(harle_screen_df.A1, harle_screen_df.A2)]

genepairs = []
for pair in list_c:
    sorted_pair = natsorted(pair)
    genepairs.append(sorted_pair)

m = []
for i in range(0 , len(genepairs)):
    a = '_'.join(genepairs[i])
    m.append(a)

harle_screen_df.insert(0, 'genepair', m, True)

In [18]:
# number of screened gene pairs
harle_screen_df.genepair.nunique()

472

In [19]:
harle_screen_df = harle_screen_df[['genepair', 'sorted_gene_pair', 'A1', 'A2', 'A1_entrez', 'A2_entrez', 
                                   'cell_line','DepMap_ID', 'mean_norm_gi', 'fdr', 'is_bassik_hit',
                                    'targetA__is_single_depleted', 'targetB__is_single_depleted',
                                    'is_sl_hit', 'targetA', 'targetB']].copy()

harle_screen_df = harle_screen_df.sort_values(['sorted_gene_pair', 'cell_line']).reset_index(drop=True)

In [20]:
# Analyze gene pair and cell line triplets in Harle Screen
print(f"Total number of rows (gene pair - cell line combinations): {len(harle_screen_df)}")
print(f"Number of unique gene pairs: {harle_screen_df['genepair'].nunique()}")
print(f"Number of unique cell lines: {harle_screen_df['cell_line'].nunique()}")
print(f"Number of unique gene pair - cell line combinations: {harle_screen_df[['genepair', 'cell_line']].drop_duplicates().shape[0]}")

harle_screen_df

Total number of rows (gene pair - cell line combinations): 12744
Number of unique gene pairs: 472
Number of unique cell lines: 27
Number of unique gene pair - cell line combinations: 12744


,genepair,sorted_gene_pair,A1,A2,A1_entrez,A2_entrez,cell_line,DepMap_ID,mean_norm_gi,fdr,is_bassik_hit,targetA__is_single_depleted,targetB__is_single_depleted,is_sl_hit,targetA,targetB
0,ABCC3_ATP5PB,ABCC3|ATP5PB,ABCC3,ATP5PB,8714.0,515.0,A-375,ACH-000219,-0.039603,0.115832,0,0,1,False,ABCC3,ATP5PB
1,ABCC3_ATP5PB,ABCC3|ATP5PB,ABCC3,ATP5PB,8714.0,515.0,A2058,ACH-000788,-0.121365,0.013146,0,0,1,False,ABCC3,ATP5PB
2,ABCC3_ATP5PB,ABCC3|ATP5PB,ABCC3,ATP5PB,8714.0,515.0,A549,ACH-000681,-0.290049,0.025119,0,0,1,False,ABCC3,ATP5PB
3,ABCC3_ATP5PB,ABCC3|ATP5PB,ABCC3,ATP5PB,8714.0,515.0,AsPC-1,ACH-000222,0.229009,0.749304,0,0,1,False,ABCC3,ATP5PB
4,ABCC3_ATP5PB,ABCC3|ATP5PB,ABCC3,ATP5PB,8714.0,515.0,BxPC-3,ACH-000535,0.090913,0.474963,0,0,1,False,ABCC3,ATP5PB
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12739,ZNF423_ZNF521,ZNF423|ZNF521,ZNF423,ZNF521,23090.0,25925.0,SK-MEL-2,ACH-001190,0.823812,1.000000,0,0,0,False,ZNF423,ZNF521
12740,ZNF423_ZNF521,ZNF423|ZNF521,ZNF423,ZNF521,23090.0,25925.0,SK-MEL-28,ACH-000615,0.403649,0.944002,0,0,0,False,ZNF423,ZNF521
12741,ZNF423_ZNF521,ZNF423|ZNF521,ZNF423,ZNF521,23090.0,25925.0,SK-MEL-5,ACH-000730,0.520101,0.983784,0,0,0,False,ZNF423,ZNF521
12742,ZNF423_ZNF521,ZNF423|ZNF521,ZNF423,ZNF521,23090.0,25925.0,SK-MES-1,ACH-000665,0.739820,1.000000,0,0,0,False,ZNF423,ZNF521


In [21]:
# save processed file
harle_screen_df.to_csv(file_path_processed_harle_df, index = False)